# Notebook 03 — Segmented Diabetes Classifier Export
Uses the frozen tongue segmentation model to create segmented crop_masked images for the diabetes classifier.
No training. No model retraining. No split modification.


In [1]:
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
import cv2

In [2]:
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy import ndimage

In [3]:
import torch
import torch.nn.functional as F
import segmentation_models_pytorch as smp

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# Paths
SPLIT_MANIFEST = Path(r'D:\DIABETES\diabetes_pipeline_outputs\02_group_safe_split_manifest.csv')
SEG_CHECKPOINT = Path(r'D:\DIABETES\Segmentation_Dataset\Segmentation_Branch_Outputs\05_training_runs\checkpoints\best_efficientnetb0_unet.pth')
OUTPUT_ROOT    = Path(r'D:\DIABETES\processed_segmented_diabetes_dataset')
PIPELINE_OUT   = Path(r'D:\DIABETES\diabetes_pipeline_outputs')
REVIEW_OUT     = PIPELINE_OUT / '03_segmented_export_review_outputs'

for p in [OUTPUT_ROOT, PIPELINE_OUT, REVIEW_OUT]:
    p.mkdir(parents=True, exist_ok=True)

for subdir in ['predicted_masks','black_background','bbox_crops','crop_masked','visual_audits','suspicious_cases']:
    (REVIEW_OUT / subdir).mkdir(exist_ok=True)

# Segmentation settings
IMG_SIZE   = 384
THRESHOLD  = 0.5
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

print('Configuration loaded.')


Device: cuda
Configuration loaded.


## Load Segmentation Model


In [4]:
model = smp.Unet(
    encoder_name="efficientnet-b0",
    encoder_weights=None,
    in_channels=3,
    classes=1,
    activation=None
)

checkpoint = torch.load(SEG_CHECKPOINT, map_location=DEVICE)

# Your checkpoint is a training checkpoint dictionary, not raw model weights.
# So we must load checkpoint["model_state_dict"].
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
else:
    state_dict = checkpoint

# Clean possible wrapper prefixes from DataParallel / torch.compile.
clean_state_dict = {}
for key, value in state_dict.items():
    new_key = key

    if new_key.startswith("module."):
        new_key = new_key.replace("module.", "", 1)

    if new_key.startswith("_orig_mod."):
        new_key = new_key.replace("_orig_mod.", "", 1)

    clean_state_dict[new_key] = value

model.load_state_dict(clean_state_dict, strict=True)
model = model.to(DEVICE)
model.eval()

print(f"Segmentation model loaded from: {SEG_CHECKPOINT}")

if isinstance(checkpoint, dict):
    print("Checkpoint keys:", list(checkpoint.keys()))

    if "epoch" in checkpoint:
        print("Checkpoint epoch:", checkpoint["epoch"])

    if "val_dice" in checkpoint:
        print("Validation Dice:", checkpoint["val_dice"])

    if "val_iou" in checkpoint:
        print("Validation IoU:", checkpoint["val_iou"])

    if "hyperparams" in checkpoint:
        print("Hyperparams:", checkpoint["hyperparams"])


Segmentation model loaded from: D:\DIABETES\Segmentation_Dataset\Segmentation_Branch_Outputs\05_training_runs\checkpoints\best_efficientnetb0_unet.pth
Checkpoint keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'val_dice', 'val_iou', 'hyperparams']
Checkpoint epoch: 40
Validation Dice: 0.9935470608749775
Validation IoU: 0.9871839441434301
Hyperparams: {'img_size': 384, 'batch_size': 8, 'lr': 0.0001, 'weight_decay': 0.0001}


## Load Split Manifest


In [5]:
df = pd.read_csv(SPLIT_MANIFEST)
print(f'Split manifest loaded: {len(df)} rows')
print(f'Columns: {list(df.columns)}')

# Detect original image path column
if 'file_path' in df.columns:
    orig_path_col = 'file_path'
elif 'original_image_path' in df.columns:
    orig_path_col = 'original_image_path'
else:
    raise ValueError('Cannot find original image path column in manifest')

print(f'Using original path column: {orig_path_col}')

required = ['final_split','final_label','label_binary']
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

# Standardize class folder names
df['class_folder'] = df['final_label'].replace({'nondiabetes':'non_diabetes'})

print('Manifest validated.')


Split manifest loaded: 2750 rows
Columns: ['image_id', 'file_path', 'dataset_source', 'original_split', 'folder_label', 'final_label', 'label_binary', 'filename', 'stem', 'extension', 'width', 'height', 'image_mode', 'file_size_bytes', 'readable', 'exact_hash_md5', 'perceptual_hash_phash', 'filename_family_id', 'exact_duplicate_group_id', 'near_duplicate_group_id', 'effective_group_id', 'audit_status', 'audit_notes', 'final_split', 'split_method', 'split_seed', 'group_size', 'group_label', 'group_status']
Using original path column: file_path
Manifest validated.


## Segmentation Inference Functions


In [6]:
def preprocess_seg(img_rgb, size=IMG_SIZE):
    img = cv2.resize(img_rgb, (size, size), interpolation=cv2.INTER_LINEAR)
    img = img.astype(np.float32) / 255.0
    for i in range(3):
        img[:,:,i] = (img[:,:,i] - MEAN[i]) / STD[i]
    return torch.from_numpy(img).permute(2,0,1).unsqueeze(0)


def run_segmentation(img_rgb):
    """Returns (mask, metrics_dict)."""
    h, w = img_rgb.shape[:2]
    start = time.perf_counter()

    inp = preprocess_seg(img_rgb).to(DEVICE)
    with torch.no_grad():
        logits = model(inp)
    prob = torch.sigmoid(logits).cpu().numpy()[0,0]

    # Resize to original size
    mask_384 = (prob > THRESHOLD).astype(np.uint8)
    mask = cv2.resize(mask_384, (w, h), interpolation=cv2.INTER_NEAREST)

    # Largest connected component
    labeled, num = ndimage.label(mask)
    if num > 0:
        sizes = ndimage.sum(mask, labeled, range(1, num+1))
        largest_label = np.argmax(sizes) + 1
        mask = (labeled == largest_label).astype(np.uint8)

    inf_ms = (time.perf_counter() - start) * 1000

    fg = mask.sum()
    fg_ratio = fg / (h * w)

    # Bbox
    if fg > 0:
        rows = np.any(mask, axis=1)
        cols = np.any(mask, axis=0)
        y1, y2 = np.where(rows)[0][[0, -1]]
        x1, x2 = np.where(cols)[0][[0, -1]]
        y2 += 1; x2 += 1
        bbox = (int(y1), int(y2), int(x1), int(x2))
        bbox_w = x2 - x1
        bbox_h = y2 - y1
        bbox_area_ratio = (bbox_w * bbox_h) / (h * w)
        touches_top = (y1 == 0)
        touches_bottom = (y2 == h)
        touches_left = (x1 == 0)
        touches_right = (x2 == w)
    else:
        bbox = None
        bbox_w = bbox_h = bbox_area_ratio = 0
        touches_top = touches_bottom = touches_left = touches_right = False

    m = {
        'mask_foreground_pixels': int(fg),
        'mask_foreground_ratio': round(float(fg_ratio), 6),
        'bbox_x1': bbox[2] if bbox else None,
        'bbox_y1': bbox[0] if bbox else None,
        'bbox_x2': bbox[3] if bbox else None,
        'bbox_y2': bbox[1] if bbox else None,
        'bbox_width': int(bbox_w),
        'bbox_height': int(bbox_h),
        'bbox_area_ratio': round(float(bbox_area_ratio), 6),
        'touches_top': touches_top,
        'touches_bottom': touches_bottom,
        'touches_left': touches_left,
        'touches_right': touches_right,
        'num_connected_components_before_lcc': num,
        'inference_ms': round(inf_ms, 2),
    }

    # Suspicious flags
    m['failed_no_mask'] = (fg == 0)
    m['suspicious_small_mask'] = (fg_ratio < 0.02) and (fg > 0)
    m['suspicious_large_mask'] = (fg_ratio > 0.85)
    m['suspicious_tiny_bbox'] = (bbox_area_ratio < 0.02) and (fg > 0)
    edge_count = sum([touches_top, touches_bottom, touches_left, touches_right])
    m['suspicious_edge_touch'] = (edge_count >= 3)

    return mask, m


def create_outputs(img_rgb, mask, bbox):
    """Returns (black_bg, bbox_crop, crop_masked)."""
    black_bg = img_rgb * mask[:,:,np.newaxis]

    if bbox:
        y1, y2, x1, x2 = bbox
        bbox_crop = img_rgb[y1:y2, x1:x2].copy()
        crop_masked = black_bg[y1:y2, x1:x2].copy()
    else:
        bbox_crop = img_rgb.copy()
        crop_masked = black_bg.copy()

    return black_bg, bbox_crop, crop_masked

print('Segmentation functions defined.')


Segmentation functions defined.


## Run Segmentation on All Images


In [7]:
records = []
excluded = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc='Segmenting'):
    orig_path = Path(row[orig_path_col])
    split = row['final_split']
    cls_folder = row['class_folder']
    img_id = row.get('image_id', f'img_{idx:06d}')
    safe_name = f"{split}_{cls_folder}_{img_id}.png"

    rec = {
        'image_id': img_id,
        'original_image_path': str(orig_path),
        'final_split': split,
        'final_label': row['final_label'],
        'label_binary': row['label_binary'],
        'effective_group_id': row.get('effective_group_id', ''),
    }

    try:
        img = cv2.imread(str(orig_path))
        if img is None:
            raise IOError('read_failed')
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        mask, metrics = run_segmentation(img_rgb)
        rec.update(metrics)

        # Decide export
        if metrics['failed_no_mask']:
            rec['segmentation_status'] = 'failed_no_mask'
            rec['export_status'] = 'excluded'
            rec['export_notes'] = 'no foreground mask'
            excluded.append(rec)
            continue

        if metrics['suspicious_small_mask'] or metrics['suspicious_tiny_bbox']:
            rec['segmentation_status'] = 'suspicious_small'
            rec['export_status'] = 'excluded'
            rec['export_notes'] = 'mask/bbox too small'
            excluded.append(rec)
            continue

        # Edge-touch alone is not exclusion
        susp_flags = []
        if metrics['suspicious_large_mask']: susp_flags.append('large_mask')
        if metrics['suspicious_edge_touch']: susp_flags.append('edge_touch')

        rec['segmentation_status'] = 'ok' if not susp_flags else '; '.join(susp_flags)
        rec['export_status'] = 'ok'
        rec['export_notes'] = ''

        # Generate outputs
        bbox = (metrics['bbox_y1'], metrics['bbox_y2'], metrics['bbox_x1'], metrics['bbox_x2']) if metrics['bbox_x1'] is not None else None
        black_bg, bbox_crop, crop_masked = create_outputs(img_rgb, mask, bbox)

        # Save to review folders
        mask_path = REVIEW_OUT / 'predicted_masks' / safe_name
        cv2.imwrite(str(mask_path), mask * 255)
        rec['predicted_mask_path'] = str(mask_path)

        bb_path = REVIEW_OUT / 'black_background' / safe_name
        cv2.imwrite(str(bb_path), cv2.cvtColor(black_bg, cv2.COLOR_RGB2BGR))
        rec['black_background_path'] = str(bb_path)

        bc_path = REVIEW_OUT / 'bbox_crops' / safe_name
        cv2.imwrite(str(bc_path), cv2.cvtColor(bbox_crop, cv2.COLOR_RGB2BGR))
        rec['bbox_crop_path'] = str(bc_path)

        cm_review_path = REVIEW_OUT / 'crop_masked' / safe_name
        cv2.imwrite(str(cm_review_path), cv2.cvtColor(crop_masked, cv2.COLOR_RGB2BGR))
        rec['crop_masked_path'] = str(cm_review_path)

        # Save to official classifier dataset
        cls_dir = OUTPUT_ROOT / split / cls_folder
        cls_dir.mkdir(parents=True, exist_ok=True)
        seg_path = cls_dir / safe_name
        cv2.imwrite(str(seg_path), cv2.cvtColor(crop_masked, cv2.COLOR_RGB2BGR))
        rec['segmented_image_path'] = str(seg_path)

        records.append(rec)

    except Exception as e:
        rec['segmentation_status'] = 'error'
        rec['export_status'] = 'failed'
        rec['export_notes'] = str(e)
        excluded.append(rec)

print(f'\nSegmentation complete.')
print(f'Exported: {len(records)}, Excluded: {len(excluded)}')


Segmenting: 100%|██████████| 2750/2750 [01:31<00:00, 29.94it/s]


Segmentation complete.
Exported: 2750, Excluded: 0


## Save Manifests and Reports


In [8]:
df_export = pd.DataFrame(records)
df_excluded = pd.DataFrame(excluded) if excluded else pd.DataFrame()

# Main manifest
df_export.to_csv(PIPELINE_OUT / '03_segmented_export_manifest.csv', index=False)
print(f'Saved: 03_segmented_export_manifest.csv ({len(df_export)} rows)')

# Summary
total = len(df)
exported = len(df_export)
excluded_count = len(excluded)

split_counts = df_export.groupby('final_split').size().to_dict()
class_counts = df_export.groupby(['final_split','final_label']).size().to_dict()

susp_small = sum(r.get('suspicious_small_mask', False) for r in excluded)
susp_large = df_export['suspicious_large_mask'].sum() if 'suspicious_large_mask' in df_export.columns else 0
susp_tiny = sum(r.get('suspicious_tiny_bbox', False) for r in excluded)
susp_edge = df_export['suspicious_edge_touch'].sum() if 'suspicious_edge_touch' in df_export.columns else 0
failed_no = sum(r.get('failed_no_mask', False) for r in excluded)

avg_inf = df_export['inference_ms'].mean() if exported > 0 else 0
med_inf = df_export['inference_ms'].median() if exported > 0 else 0

summary = pd.DataFrame([{
    'total_input_images': total,
    'total_exported': exported,
    'total_excluded': excluded_count,
    'total_failed_inference': failed_no,
    **{f'split_{k}': v for k, v in split_counts.items()},
    **{f'{sp}_{cl}': v for (sp, cl), v in class_counts.items()},
    'suspicious_small_mask': susp_small,
    'suspicious_large_mask': int(susp_large),
    'suspicious_tiny_bbox': susp_tiny,
    'suspicious_edge_touch': int(susp_edge),
    'failed_no_mask': failed_no,
    'avg_inference_ms': round(avg_inf, 2),
    'median_inference_ms': round(med_inf, 2),
}])
summary.to_csv(PIPELINE_OUT / '03_segmented_export_summary.csv', index=False)
print('Saved: 03_segmented_export_summary.csv')

# Suspicious cases
if not df_excluded.empty:
    df_excluded.to_csv(PIPELINE_OUT / '03_segmented_suspicious_cases.csv', index=False)
    print(f'Saved: 03_segmented_suspicious_cases.csv ({len(df_excluded)} cases)')
else:
    pd.DataFrame().to_csv(PIPELINE_OUT / '03_segmented_suspicious_cases.csv', index=False)
    print('No suspicious cases — empty CSV saved.')


Saved: 03_segmented_export_manifest.csv (2750 rows)
Saved: 03_segmented_export_summary.csv
No suspicious cases — empty CSV saved.


## Visual Audit Grid


In [9]:
n = min(16, len(df_export))
sample = df_export.sample(n, random_state=42)

fig, axes = plt.subplots(n, 4, figsize=(16, n*3))
if n == 1: axes = axes[np.newaxis, :]
fig.suptitle('Segmentation Visual Audit (Random Sample)', fontsize=14)

col_titles = ['Original', 'Mask', 'Crop Masked', 'Bbox Crop']
for j, t in enumerate(col_titles):
    axes[0, j].set_title(t, fontsize=10)

for i, (_, row) in enumerate(sample.iterrows()):
    try:
        orig = cv2.imread(row['original_image_path'])
        orig = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(row['predicted_mask_path'], cv2.IMREAD_GRAYSCALE)
        crop_masked = cv2.imread(row['crop_masked_path'])
        crop_masked = cv2.cvtColor(crop_masked, cv2.COLOR_BGR2RGB)
        bbox_crop = cv2.imread(row['bbox_crop_path'])
        bbox_crop = cv2.cvtColor(bbox_crop, cv2.COLOR_BGR2RGB)

        axes[i,0].imshow(orig)
        axes[i,1].imshow(mask, cmap='gray')
        axes[i,2].imshow(crop_masked)
        axes[i,3].imshow(bbox_crop)
    except Exception:
        pass

    for j in range(4):
        axes[i,j].axis('off')

plt.tight_layout()
plt.savefig(PIPELINE_OUT / '03_segmented_visual_audit_grid.png', dpi=100)
plt.close()
print('Saved: 03_segmented_visual_audit_grid.png')


Saved: 03_segmented_visual_audit_grid.png


## Export Report and Status


In [10]:
# Determine status
if exported == 0:
    status = 'FAIL'
elif excluded_count > 0 or susp_edge > 0:
    status = 'PASS WITH WARNING'
else:
    status = 'PASS'

report = f"""================================================
NOTEBOOK 03 — SEGMENTED EXPORT REPORT
================================================

STATUS: {status}

Total input images    : {total}
Exported images       : {exported}
Excluded images       : {excluded_count}
Failed (no mask)      : {failed_no}

Suspicious counts:
  Small mask          : {susp_small}
  Large mask          : {int(susp_large)}
  Tiny bbox           : {susp_tiny}
  Edge touch (3+ edges): {int(susp_edge)}

Split counts:
  {' | '.join(f'{k}={v}' for k, v in split_counts.items())}

Class counts:
  {' | '.join(f'{sp}/{cl}={v}' for (sp, cl), v in class_counts.items())}

Inference performance:
  Average: {avg_inf:.2f} ms/image
  Median:  {med_inf:.2f} ms/image

Output dataset root:
  {OUTPUT_ROOT}

Manifest path:
  {PIPELINE_OUT / '03_segmented_export_manifest.csv'}

================================================
IMPORTANT
================================================

This is the segmented classifier export, NOT classifier training.

Files 04–07 must use:
  D:\DIABETES\diabetes_pipeline_outputs\03_segmented_export_manifest.csv

NOT:
  D:\DIABETES\diabetes_pipeline_outputs\03_export_manifest.csv

The segmented crop_masked images in:
  {OUTPUT_ROOT}
are the official classifier inputs.

Review outputs saved to:
  {REVIEW_OUT}

================================================
"""

with open(PIPELINE_OUT / '03_segmented_export_report.txt', 'w') as f:
    f.write(report)

print(report)


NOTEBOOK 03 — SEGMENTED EXPORT REPORT

STATUS: PASS WITH WARNING

Total input images    : 2750
Exported images       : 2750
Excluded images       : 0
Failed (no mask)      : 0

Suspicious counts:
  Small mask          : 0
  Large mask          : 45
  Tiny bbox           : 0
  Edge touch (3+ edges): 554

Split counts:
  test=412 | train=1930 | val=408

Class counts:
  test/diabetes=215 | test/non_diabetes=197 | train/diabetes=961 | train/non_diabetes=969 | val/diabetes=199 | val/non_diabetes=209

Inference performance:
  Average: 12.83 ms/image
  Median:  10.65 ms/image

Output dataset root:
  D:\DIABETES\processed_segmented_diabetes_dataset

Manifest path:
  D:\DIABETES\diabetes_pipeline_outputs\03_segmented_export_manifest.csv

IMPORTANT

This is the segmented classifier export, NOT classifier training.

Files 04–07 must use:
  D:\DIABETES\diabetes_pipeline_outputs_segmented_export_manifest.csv

NOT:
  D:\DIABETES\diabetes_pipeline_outputs_export_manifest.csv

The segmented crop_mas

<>:62: SyntaxWarning: invalid escape sequence '\D'
<>:62: SyntaxWarning: invalid escape sequence '\D'
C:\Users\CompuMark\AppData\Local\Temp\ipykernel_28020\2860573359.py:62: SyntaxWarning: invalid escape sequence '\D'
  """
